# FedFalsify v0.6 — One-Click Resumable Confirmatory Experiment

This notebook clones the frozen GitHub branch, mounts Google Drive, runs the
confirmatory experiment in deterministic chunks, seals every chunk with hashes
and a source-code fingerprint, mirrors outputs to Drive, and optionally commits
the outputs back to GitHub.

**Recommended mode:** `run_all_and_merge`. If Colab disconnects, run the notebook
again with the same configuration. Completed chunks are restored from Drive and skipped.


In [ ]:
#@title 1. Configuration
REPO_URL = "https://github.com/AzizulHakim00/fedfalsify.git" #@param {type:"string"}
CODE_BRANCH = "feat/fedfalsify-mvi" #@param {type:"string"}
PUSH_BRANCH = "feat/fedfalsify-mvi" #@param {type:"string"}
RUN_ID = "v06-primary-confirmatory" #@param {type:"string"}

MODE = "dry_run" #@param ["dry_run", "run_one_chunk", "run_all_and_merge", "merge_only"]
CHUNK_INDEX = 0 #@param {type:"integer"}
TOTAL_CHUNKS = 4 #@param {type:"integer"}

DRIVE_ROOT = "/content/drive/MyDrive/FedFalsify/results" #@param {type:"string"}
PUSH_TO_GITHUB = True #@param {type:"boolean"}
PUSH_DRY_RUN = False #@param {type:"boolean"}
PUSH_AFTER_EACH_CHUNK = True #@param {type:"boolean"}

GIT_USER_NAME = "Azizul Hakim" #@param {type:"string"}
GIT_USER_EMAIL = "AzizulHakim00@users.noreply.github.com" #@param {type:"string"}

assert MODE in {"dry_run", "run_one_chunk", "run_all_and_merge", "merge_only"}
assert TOTAL_CHUNKS == 4, "The frozen primary protocol uses exactly four chunks."
assert 0 <= CHUNK_INDEX < TOTAL_CHUNKS
print({
    "mode": MODE,
    "run_id": RUN_ID,
    "chunk_index": CHUNK_INDEX,
    "total_chunks": TOTAL_CHUNKS,
    "drive_root": DRIVE_ROOT,
})


In [ ]:
#@title 2. Mount Drive, clone repository, restore backups, and install
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_DIR = Path("/content/fedfalsify")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", CODE_BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", CODE_BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", CODE_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    os.chdir(REPO_DIR)

repo_output_root = REPO_DIR / "results" / "colab"
drive_output_root = Path(DRIVE_ROOT)
drive_output_root.mkdir(parents=True, exist_ok=True)

active_run_id = f"{RUN_ID}-dry-run" if MODE == "dry_run" else RUN_ID
drive_run = drive_output_root / active_run_id
repo_run = repo_output_root / active_run_id
if drive_run.exists():
    shutil.copytree(drive_run, repo_run, dirs_exist_ok=True)
    print("Restored prior Drive backup:", drive_run)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run(["git", "config", "user.name", GIT_USER_NAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_USER_EMAIL], check=True)

print("Repository:", REPO_DIR)
print("Code commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(["fedfalsify-colab", "--help"], check=True)
subprocess.run(["fedfalsify-colab-audit", "--help"], check=True)


In [ ]:
#@title 3. Validate Colab-specific code and notebook structure
import subprocess
subprocess.run(
    [
        "pytest", "-q",
        "tests/test_colab_pipeline.py",
        "tests/test_colab_audit.py",
        "tests/test_colab_auto_notebook.py",
    ],
    check=True,
    cwd=REPO_DIR,
)
print("Colab preflight validation passed.")


In [ ]:
#@title 4. Secure GitHub result push helper
import getpass
import os
from pathlib import Path
import stat
import subprocess

def _github_token():
    try:
        from google.colab import userdata
        value = userdata.get("GITHUB_TOKEN")
    except Exception:
        value = None
    if not value:
        value = getpass.getpass(
            "GitHub fine-grained token (Contents: Read and write): "
        )
    if not value:
        raise RuntimeError("A GitHub token is required for result push.")
    return value

def push_run_results(active_run_id, message):
    relative_output = Path("results") / "colab" / active_run_id
    subprocess.run(["git", "add", "--", str(relative_output)], check=True, cwd=REPO_DIR)
    staged = subprocess.run(
        ["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR
    ).returncode != 0
    if staged:
        subprocess.run(["git", "commit", "-m", message], check=True, cwd=REPO_DIR)
    else:
        print("No new Git-tracked result files to commit.")
        return

    token = _github_token()
    askpass = Path("/tmp/fedfalsify_git_askpass.sh")
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GITHUB_TOKEN"] = token
    try:
        subprocess.run(
            ["git", "pull", "--rebase", "origin", PUSH_BRANCH],
            check=True, cwd=REPO_DIR, env=env,
        )
        subprocess.run(
            ["git", "push", "origin", f"HEAD:{PUSH_BRANCH}"],
            check=True, cwd=REPO_DIR, env=env,
        )
    finally:
        askpass.unlink(missing_ok=True)
        env.pop("GITHUB_TOKEN", None)
        del token
    print("Pushed result commit to:", PUSH_BRANCH)


In [ ]:
#@title 5. Resumable experiment helpers
from pathlib import Path
import shutil
import subprocess

PRIMARY_BENCHMARKS = "base,poly3,nested_sine,trig_product,interaction"
PRIMARY_SCENARIOS = "complementary,spurious,exception"
PRIMARY_NOISE = "0.03,0.10"
PRIMARY_SEEDS = "9001-9020"

def run_primary_chunk(index):
    command = [
        "fedfalsify-colab", "run-chunk",
        "--run-id", RUN_ID,
        "--output-root", str(repo_output_root),
        "--drive-root", str(drive_output_root),
        "--benchmarks", PRIMARY_BENCHMARKS,
        "--scenarios", PRIMARY_SCENARIOS,
        "--noise", PRIMARY_NOISE,
        "--samples", "300",
        "--clients", "4",
        "--seeds", PRIMARY_SEEDS,
        "--chunk-index", str(index),
        "--total-chunks", str(TOTAL_CHUNKS),
        "--population-size", "48",
        "--generations", "12",
        "--max-genes", "4",
        "--bootstrap-resamples", "4000",
    ]
    subprocess.run(command, check=True, cwd=REPO_DIR)
    subprocess.run(
        [
            "fedfalsify-colab-audit", "seal-chunk",
            "--repo-root", str(REPO_DIR),
            "--run-id", RUN_ID,
            "--output-root", str(repo_output_root),
            "--drive-root", str(drive_output_root),
            "--chunk-index", str(index),
            "--total-chunks", str(TOTAL_CHUNKS),
        ],
        check=True, cwd=REPO_DIR,
    )
    if PUSH_TO_GITHUB and PUSH_AFTER_EACH_CHUNK:
        push_run_results(
            RUN_ID,
            f"results: save confirmatory chunk {index + 1} of {TOTAL_CHUNKS}",
        )

def verify_merge_and_seal(run_id, expected_chunks, expected_rows, bootstrap_resamples):
    subprocess.run(
        [
            "fedfalsify-colab-audit", "verify-run",
            "--run-id", run_id,
            "--output-root", str(repo_output_root),
            "--drive-root", str(drive_output_root),
            "--expected-chunks", str(expected_chunks),
            "--expected-rows", str(expected_rows),
        ],
        check=True, cwd=REPO_DIR,
    )
    subprocess.run(
        [
            "fedfalsify-colab", "merge",
            "--run-id", run_id,
            "--output-root", str(repo_output_root),
            "--drive-root", str(drive_output_root),
            "--expected-chunks", str(expected_chunks),
            "--expected-rows", str(expected_rows),
            "--bootstrap-resamples", str(bootstrap_resamples),
        ],
        check=True, cwd=REPO_DIR,
    )
    subprocess.run(
        [
            "fedfalsify-colab-audit", "seal-final",
            "--run-id", run_id,
            "--output-root", str(repo_output_root),
            "--drive-root", str(drive_output_root),
            "--expected-rows", str(expected_rows),
        ],
        check=True, cwd=REPO_DIR,
    )


In [ ]:
#@title 6. Execute selected mode
import subprocess

if MODE == "dry_run":
    dry_run_id = f"{RUN_ID}-dry-run"
    subprocess.run(
        [
            "fedfalsify-colab", "run-chunk",
            "--run-id", dry_run_id,
            "--output-root", str(repo_output_root),
            "--drive-root", str(drive_output_root),
            "--benchmarks", "base",
            "--scenarios", "complementary",
            "--noise", "0.03",
            "--samples", "60",
            "--clients", "4",
            "--seeds", "9001",
            "--chunk-index", "0",
            "--total-chunks", "1",
            "--population-size", "12",
            "--generations", "2",
            "--max-genes", "3",
            "--bootstrap-resamples", "500",
        ],
        check=True, cwd=REPO_DIR,
    )
    subprocess.run(
        [
            "fedfalsify-colab-audit", "seal-chunk",
            "--repo-root", str(REPO_DIR),
            "--run-id", dry_run_id,
            "--output-root", str(repo_output_root),
            "--drive-root", str(drive_output_root),
            "--chunk-index", "0",
            "--total-chunks", "1",
        ],
        check=True, cwd=REPO_DIR,
    )
    verify_merge_and_seal(dry_run_id, 1, 4, 500)
    if PUSH_TO_GITHUB and PUSH_DRY_RUN:
        push_run_results(dry_run_id, "results: save Colab confirmatory dry run")

elif MODE == "run_one_chunk":
    run_primary_chunk(CHUNK_INDEX)

elif MODE == "run_all_and_merge":
    for index in range(TOTAL_CHUNKS):
        print(f"\n=== Primary chunk {index + 1}/{TOTAL_CHUNKS} ===")
        run_primary_chunk(index)
    verify_merge_and_seal(RUN_ID, TOTAL_CHUNKS, 2400, 10000)
    if PUSH_TO_GITHUB:
        push_run_results(RUN_ID, "results: seal complete v0.6 confirmatory matrix")

elif MODE == "merge_only":
    verify_merge_and_seal(RUN_ID, TOTAL_CHUNKS, 2400, 10000)
    if PUSH_TO_GITHUB:
        push_run_results(RUN_ID, "results: seal complete v0.6 confirmatory matrix")

print("Completed mode:", MODE)


In [ ]:
#@title 7. Inspect verified outputs
from pathlib import Path
import json

active_run_id = f"{RUN_ID}-dry-run" if MODE == "dry_run" else RUN_ID
repo_run = REPO_DIR / "results" / "colab" / active_run_id
drive_run = Path(DRIVE_ROOT) / active_run_id

print("Git working-tree output:", repo_run)
print("Drive mirror:", drive_run)
for path in sorted(repo_run.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO_DIR), path.stat().st_size, "bytes")

verified = repo_run / "VERIFIED.json"
if verified.exists():
    print("\nVERIFIED:")
    print(verified.read_text(encoding="utf-8"))

final_summary = repo_run / "final" / "v06_confirmatory_holm.json"
if final_summary.exists():
    report = json.loads(final_summary.read_text(encoding="utf-8"))
    print("\nMethod summary:")
    print(json.dumps(report.get("methods", {}), indent=2))
    print("\nMultiple testing:")
    print(json.dumps(report.get("multiple_testing", {}), indent=2))


## How to use

1. Run once with `MODE="dry_run"`.
2. Then set `MODE="run_all_and_merge"` and run all cells.
3. If Colab disconnects, run the same notebook again. Drive restoration plus
   chunk manifests prevent completed chunks from being rerun.
4. For shorter sessions use `MODE="run_one_chunk"` with indexes `0`, `1`, `2`,
   and `3`, then use `MODE="merge_only"`.
5. Confirm that both GitHub and Drive contain `COMPLETE` and `VERIFIED.json`.

For GitHub push, add the Colab secret `GITHUB_TOKEN`, give the notebook access,
and use a fine-grained token limited to this repository with **Contents: Read and
write** permission. The token is never saved in Git or Drive.
